# Adoption of AI Among Students

This analysis evaluates patterns of AI usage, adoption behavior, satisfaction, and continued usage across student groups.

Key focus:
- Adoption behavior
- Usage intensity
- Satisfaction trends
- Retention patterns


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

df = pd.read_csv('assets/ai_assistant_usage_student_life.csv')

df.head()

## Data Overview

Initial inspection of structure, variable types, and summary statistics.

In [ ]:
print('Shape:', df.shape)
df.info()
df.describe()
df.describe(include='object')

## Data Cleaning and Preparation

In [ ]:
# Remove duplicates
df.drop_duplicates(inplace=True)

# Handle missing values
for col in df.select_dtypes(include=['number']):
   df[col] = df[col].fillna(df[col].median())

# Convert date
df['SessionDate'] = pd.to_datetime(df['SessionDate'], errors='coerce')

## Feature Engineering

In [ ]:
# Adoption indicator
df['HighUsage'] = df['TotalPrompts'] > df['TotalPrompts'].median()

# Efficiency metric
df['Efficiency'] = df['TotalPrompts'] / df['SessionLengthMin']

# Monthly usage
df['Month'] = df['SessionDate'].dt.month

## Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))

sns.histplot(df['SessionLengthMin'], bins=20, ax=axes[0])
axes[0].set_title('Session Length')

sns.histplot(df['TotalPrompts'], bins=20, ax=axes[1])
axes[1].set_title('Total Prompts')

sns.histplot(df['SatisfactionRating'], bins=10, ax=axes[2])
axes[2].set_title('Satisfaction')

plt.tight_layout()
plt.show()

## Adoption Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,5))

sns.barplot(x='StudentLevel', y='HighUsage', hue='UsedAgain', palette='Set1', data=df, ax=axes[0])
axes[0].set_title('Adoption by Student Level')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(x='Discipline', y='HighUsage', hue='UsedAgain', palette='Set1', data=df, ax=axes[1])
axes[1].set_title('Adoption by Discipline')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Usage and Behavior

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,5))

sns.scatterplot(x='SessionLengthMin', y='TotalPrompts', hue='StudentLevel', palette='Set1', data=df, ax=axes[0])
axes[0].set_title('Session vs Prompts')

sns.violinplot(x='StudentLevel', y='TotalPrompts', data=df, ax=axes[1])
axes[1].set_title('Prompt Distribution')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Satisfaction Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,5))

sns.boxplot(x='StudentLevel', y='SatisfactionRating', hue='UsedAgain', palette='Set1', data=df, ax=axes[0])
axes[0].set_title('Satisfaction by Level')
axes[0].tick_params(axis='x', rotation=30)

sns.scatterplot(x='Efficiency', y='SatisfactionRating', hue='HighUsage', palette='Set1', data=df, ax=axes[1])
axes[1].set_title('Efficiency vs Satisfaction')

plt.tight_layout()
plt.show()

## Time Trend

In [ ]:
df.groupby('Month')['TotalPrompts'].mean().plot(figsize=(8,4))
plt.title('Monthly Usage Trend')
plt.show()

## Correlation

In [ ]:
plt.figure(figsize=(8,5))
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()



## Key Insights

- Students who use AI more tend to spend more time per session
- Higher usage is generally linked with higher satisfaction
- Some student groups use AI more frequently than others
- Students who use AI more are more likely to use it again
- Usage patterns differ across disciplines


In [ ]:
pivot = df.pivot_table(values='HighUsage', index='StudentLevel', columns='Discipline', aggfunc='mean')

plt.figure(figsize=(10,6))
sns.heatmap(pivot, annot=True, cmap='coolwarm')
plt.title('Adoption Across Groups')
plt.show()

## Logistic Regression

A simple classification model is used to predict whether a student will continue using AI tools based on their usage and behavior.

### Insight
Students with higher usage and satisfaction are more likely to continue using AI, showing a strong link between engagement and adoption.

In [ ]:
# Logistic Regression Model

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Copy dataset
df_ml = df.copy()

# Encode categorical columns
for col in df_ml.select_dtypes(include='object'):
    df_ml[col] = LabelEncoder().fit_transform(df_ml[col].astype(str))

# Convert boolean to int
df_ml['UsedAgain'] = df_ml['UsedAgain'].astype(int)
df_ml['HighUsage'] = df_ml['HighUsage'].astype(int)

# Drop unnecessary columns
X = df_ml.drop(['UsedAgain', 'SessionID', 'SessionDate'], axis=1)
y = df_ml['UsedAgain']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Results
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion Matrix
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
plt.title("Confusion Matrix")
plt.show()